In [68]:
# I will remove the comments because I can
# I'm a Django Back-End Developer idk why I'm here honestly

import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore') # yes don't ask

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/ioai-philippines-2026-round-2-semi-finals/ioai_qc_train.csv
/kaggle/input/competitions/ioai-philippines-2026-round-2-semi-finals/SUBMISSION_FINAL_INT.csv
/kaggle/input/competitions/ioai-philippines-2026-round-2-semi-finals/TEST_FINAL_INT.csv


In [69]:
train = pd.read_csv('/kaggle/input/competitions/ioai-philippines-2026-round-2-semi-finals/ioai_qc_train.csv')
test = pd.read_csv('/kaggle/input/competitions/ioai-philippines-2026-round-2-semi-finals/TEST_FINAL_INT.csv')

print("LOAD DATA PLS")
print("data shape:", train.shape)
print("columns:", train.columns.tolist())
print(f"Missing temp data: {train['temperature_2m'].isnull().sum()}")

LOAD DATA PLS
data shape: (26304, 44)
columns: ['city_name', 'datetime', 'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'apparent_temperature', 'precipitation', 'rain', 'snowfall', 'snow_depth', 'weather_code', 'pressure_msl', 'surface_pressure', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'et0_fao_evapotranspiration', 'vapour_pressure_deficit', 'wind_speed_10m', 'wind_speed_100m', 'wind_direction_10m', 'wind_direction_100m', 'wind_gusts_10m', 'soil_temperature_0_to_7cm', 'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 'soil_temperature_100_to_255cm', 'soil_moisture_0_to_7cm', 'soil_moisture_7_to_28cm', 'soil_moisture_28_to_100cm', 'soil_moisture_100_to_255cm', 'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'direct_normal_irradiance', 'global_tilted_irradiance', 'terrestrial_radiation', 'shortwave_radiation_instant', 'direct_radiation_instant', 'diffuse_radiation_instant', 'direct_normal_irradiance_instant', 'global

In [70]:
# data cleaning for temp ig why are there 1282 missing temp values why
train['temperature_2m'] = train['temperature_2m'].fillna(method='ffill').fillna(method='bfill')
test['temperature_2m'] = test['temperature_2m'].fillna(method='ffill').fillna(method='bfill')

# in case of emergency
train['temperature_2m'].fillna(train['temperature_2m'].median(), inplace=True)
test['temperature_2m'].fillna(train['temperature_2m'].median(), inplace=True)

# fix impossible sensor values bruh 
train['relative_humidity_2m'] = train['relative_humidity_2m'].clip(0, 100)
test['relative_humidity_2m'] = test['relative_humidity_2m'].clip(0, 100)

# random missing values
numeric_cols = train.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if col != 'precipitation': 
        median_val = train[col].median()
        train[col].fillna(median_val, inplace=True)
        if col in test.columns:
            test[col].fillna(median_val, inplace=True)

print(f"Missing values: {train.isnull().sum().sum()}")
# ok missing values = 0 pog

Missing values: 0


In [71]:
# creating features
def create_features(df):
    df = df.copy()

    # time for me to go to sleep and play League of Legends
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['hour'] = df['datetime'].dt.hour
    df['day_of_week'] = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month
    df['day_of_year'] = df['datetime'].dt.dayofyear
    df['week_of_year'] = df['datetime'].dt.isocalendar().week    
    
    # cycles wow
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)
    df['month_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 12)
    
    # I LOVE BAGYO NO CLASS
    df['is_rainy_season'] = df['month'].isin([6, 7, 8, 9, 10, 11]).astype(int)
    df['is_morning'] = ((df['hour'] >= 6) & (df['hour'] < 12)).astype(int)
    df['is_afternoon'] = ((df['hour'] >= 12) & (df['hour'] < 18)).astype(int)
    df['is_evening'] = ((df['hour'] >= 18) & (df['hour'] < 24)).astype(int)
    df['is_night'] = ((df['hour'] >= 0) & (df['hour'] < 6)).astype(int)
    
    df['temp_humidity'] = df['temperature_2m'] * df['relative_humidity_2m']
    df['dew_point_spread'] = df['temperature_2m'] - df['dew_point_2m']
    df['apparent_temp_diff'] = df['temperature_2m'] - df['apparent_temperature']
    df['cloud_humidity'] = df['cloud_cover'] * df['relative_humidity_2m']
    df['wind_effect'] = df['wind_speed_10m'] * (df['temperature_2m'] - df['apparent_temperature'])
    df['wind_chill'] = df['wind_speed_10m'] * df['dew_point_spread']

    df['pressure_change_1h'] = df['pressure_msl'].diff().fillna(0)
    df['temp_change_1h'] = df['temperature_2m'].diff().fillna(0)
    df['humidity_change_1h'] = df['relative_humidity_2m'].diff().fillna(0)
    df['wind_change_1h'] = df['wind_speed_10m'].diff().fillna(0)
    
    df['temp_rolling_3h'] = df['temperature_2m'].rolling(3, min_periods=1).mean()
    df['temp_rolling_6h'] = df['temperature_2m'].rolling(6, min_periods=1).mean()
    df['temp_rolling_12h'] = df['temperature_2m'].rolling(12, min_periods=1).mean()
    df['temp_std_6h'] = df['temperature_2m'].rolling(6, min_periods=1).std().fillna(0)
    df['temp_std_12h'] = df['temperature_2m'].rolling(12, min_periods=1).std().fillna(0)
    
    df['pressure_rolling_3h'] = df['pressure_msl'].rolling(3, min_periods=1).mean()
    df['pressure_rolling_6h'] = df['pressure_msl'].rolling(6, min_periods=1).mean()
    df['pressure_rolling_12h'] = df['pressure_msl'].rolling(12, min_periods=1).mean()
    df['pressure_std_6h'] = df['pressure_msl'].rolling(6, min_periods=1).std().fillna(0)
    df['pressure_change_3h'] = df['pressure_msl'] - df['pressure_rolling_3h']
    df['pressure_change_6h'] = df['pressure_msl'] - df['pressure_rolling_6h']
    
    df['humidity_rolling_3h'] = df['relative_humidity_2m'].rolling(3, min_periods=1).mean()
    df['humidity_rolling_6h'] = df['relative_humidity_2m'].rolling(6, min_periods=1).mean()
    df['humidity_rolling_12h'] = df['relative_humidity_2m'].rolling(12, min_periods=1).mean()
    df['humidity_std_6h'] = df['relative_humidity_2m'].rolling(6, min_periods=1).std().fillna(0)
    
    df['wind_rolling_3h'] = df['wind_speed_10m'].rolling(3, min_periods=1).mean()
    df['wind_rolling_6h'] = df['wind_speed_10m'].rolling(6, min_periods=1).mean()
    df['wind_rolling_12h'] = df['wind_speed_10m'].rolling(12, min_periods=1).mean()
    df['wind_std_6h'] = df['wind_speed_10m'].rolling(6, min_periods=1).std().fillna(0)
    
    df['cloud_rolling_3h'] = df['cloud_cover'].rolling(3, min_periods=1).mean()
    df['cloud_rolling_6h'] = df['cloud_cover'].rolling(6, min_periods=1).mean()
    df['cloud_rolling_12h'] = df['cloud_cover'].rolling(12, min_periods=1).mean()
    
    df['dew_rolling_3h'] = df['dew_point_2m'].rolling(3, min_periods=1).mean()
    df['dew_rolling_6h'] = df['dew_point_2m'].rolling(6, min_periods=1).mean()
    
    return df
# I gave up sa comments

# Feature Enjinyiring
train = create_features(train)
test = create_features(test)

In [72]:
# models.py in django
features = [
    'temperature_2m',
    'relative_humidity_2m',
    'dew_point_2m',
    'apparent_temperature',
    'pressure_msl',
    'surface_pressure',
    'wind_speed_10m',
    'wind_speed_100m',
    'wind_direction_10m',
    'wind_gusts_10m',
    'cloud_cover',
    'cloud_cover_low',
    'cloud_cover_mid',
    'cloud_cover_high',
    'weather_code',
    'vapour_pressure_deficit',
    'soil_moisture_0_to_7cm',
    'soil_moisture_7_to_28cm',
    'hour',
    'day_of_week',
    'month',
    'day_of_year',
    'week_of_year',
    'hour_sin',
    'hour_cos',
    'day_sin',
    'day_cos',
    'month_sin',
    'month_cos',
    'is_rainy_season',
    'is_morning',
    'is_afternoon',
    'is_evening',
    'is_night',
    'temp_humidity',
    'dew_point_spread',
    'apparent_temp_diff',
    'cloud_humidity',
    'wind_chill',    
    'temp_rolling_3h',
    'temp_rolling_6h',
    'temp_rolling_12h',
    'temp_std_6h',
    'temp_std_12h',    
    'pressure_rolling_3h',
    'pressure_rolling_6h',
    'pressure_rolling_12h',
    'pressure_std_6h',
    'pressure_change_3h',
    'pressure_change_6h',
    'humidity_rolling_3h',
    'humidity_rolling_6h',
    'humidity_rolling_12h',
    'humidity_std_6h',    
    'wind_rolling_3h',
    'wind_rolling_6h',
    'wind_rolling_12h',
    'wind_std_6h',    
    'cloud_rolling_3h',
    'cloud_rolling_6h',
    'cloud_rolling_12h',
    'dew_rolling_3h',
    'dew_rolling_6h',
    'pressure_change_1h',
    'temp_change_1h',
    'humidity_change_1h',
    'wind_change_1h',
]

features = [f for f in features if f in train.columns and f in test.columns]

X_train = train[features]
y_train = train['precipitation']

model = XGBRegressor(
    n_estimators=250,        
    learning_rate=0.025,      
    max_depth=6,             
    min_child_weight=3,      
    subsample=0.85,           
    colsample_bytree=0.85,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)

model.fit(X_train, y_train)

feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

In [73]:
# this is the one frfrfrfrfrfrfrfrfr
test_preds = model.predict(test[features])
test_preds = np.maximum(0, test_preds)

submission = pd.DataFrame({
    'id': test['id'],
    'precipitation': test_preds
})

submission.to_csv('this_the_one_fr_submission_v5.csv', index=False)